In [2]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

df = pd.read_csv("data/raw/indian_roads_dataset.csv")

print("Shape:", df.shape)
df.head()

Shape: (20000, 24)


,accident_id,city,state,latitude,longitude,date,time,hour,day_of_week,is_weekend,road_type,lanes,traffic_signal,weather,visibility,temperature,traffic_density,cause,accident_severity,vehicles_involved,casualties,is_peak_hour,festival,risk_score
0,0,Pune,Maharashtra,18.680827,73.930388,2023-10-22,5:00,5,Sunday,1,highway,3,1,fog,low,32,high,weather,fatal,2,2,0,NaN,0.85
1,1,Mumbai,Maharashtra,18.817732,72.790846,2023-05-21,4:00,4,Sunday,1,urban,4,0,clear,high,34,low,weather,major,4,3,0,NaN,0.10
2,2,Mumbai,Maharashtra,19.096889,72.819424,2024-07-10,13:00,13,Wednesday,0,urban,3,0,fog,low,21,medium,weather,minor,1,1,0,NaN,0.45
3,3,Chandigarh,Punjab,30.787805,76.847507,2025-03-30,11:00,11,Sunday,1,urban,1,1,fog,low,30,high,distraction,minor,5,2,0,NaN,0.65
4,4,Chennai,Tamil Nadu,12.965155,80.283313,2024-01-25,16:00,16,Thursday,0,highway,3,1,clear,high,24,low,distraction,minor,2,1,0,NaN,0.10


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 24 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   accident_id        20000 non-null  int64  
 1   city               20000 non-null  object 
 2   state              20000 non-null  object 
 3   latitude           20000 non-null  float64
 4   longitude          20000 non-null  float64
 5   date               20000 non-null  object 
 6   time               20000 non-null  object 
 7   hour               20000 non-null  int64  
 8   day_of_week        20000 non-null  object 
 9   is_weekend         20000 non-null  int64  
 10  road_type          20000 non-null  object 
 11  lanes              20000 non-null  int64  
 12  traffic_signal     20000 non-null  int64  
 13  weather            20000 non-null  object 
 14  visibility         20000 non-null  object 
 15  temperature        20000 non-null  int64  
 16  traffic_density    200

In [4]:
df.describe

<bound method NDFrame.describe of        accident_id        city        state   latitude  longitude        date  \
0                0        Pune  Maharashtra  18.680827  73.930388  2023-10-22   
1                1      Mumbai  Maharashtra  18.817732  72.790846  2023-05-21   
2                2      Mumbai  Maharashtra  19.096889  72.819424  2024-07-10   
3                3  Chandigarh       Punjab  30.787805  76.847507  2025-03-30   
4                4     Chennai   Tamil Nadu  12.965155  80.283313  2024-01-25   
...            ...         ...          ...        ...        ...         ...   
19995        19995   Bangalore    Karnataka  13.092276  77.599571  2022-09-29   
19996        19996     Chennai   Tamil Nadu  13.172928  80.157062  2023-11-25   
19997        19997     Chennai   Tamil Nadu  12.997170  80.150724  2022-06-18   
19998        19998     Kolkata  West Bengal  22.454882  88.322213  2023-03-12   
19999        19999       Delhi        Delhi  28.510266  77.065301  2024-07-

In [5]:
missing = df.isnull().sum().sort_values(ascending=False)
missing[missing > 0]

festival    19885
dtype: int64

In [10]:
df[["latitude", "longitude"]].isnull().sum()


latitude     0
longitude    0
dtype: int64

In [12]:
df = df[
    (df["latitude"].between(-90, 90)) &
    (df["longitude"].between(-180, 180))
]

In [14]:
df[["latitude", "longitude"]].describe()


,latitude,longitude
count,20000.000000,20000.000000
mean,20.389207,78.173330
std,6.165791,4.485967
min,12.800172,72.700017
25%,13.198653,73.997979
50%,18.812008,77.297000
75%,28.402467,80.111089
max,30.799960,88.499861


In [15]:
df.duplicated().sum()

np.int64(0)

In [17]:
df["date"] = pd.to_datetime(df["date"], errors="coerce")

In [19]:
df["date"].isnull().sum()

np.int64(0)

In [20]:
df["time"] = pd.to_datetime(df["time"], errors="coerce").dt.time

/var/folders/wq/qjtlzxqd7230xkvd70xf0jdr0000gn/T/ipykernel_23428/3691430095.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["time"] = pd.to_datetime(df["time"], errors="coerce").dt.time


In [21]:
df["datetime"] = pd.to_datetime(
    df["date"].astype(str) + " " + df["time"].astype(str),
    errors="coerce"
)

In [22]:
df = df.dropna(subset=["datetime"])

In [23]:
df["datetime"].head()

0   2023-10-22 05:00:00
1   2023-05-21 04:00:00
2   2024-07-10 13:00:00
3   2025-03-30 11:00:00
4   2024-01-25 16:00:00
Name: datetime, dtype: datetime64[ns]

In [24]:
df["year"] = df["datetime"].dt.year
df["month"] = df["datetime"].dt.month
df["day"] = df["datetime"].dt.day
df["hour"] = df["datetime"].dt.hour
df["weekday"] = df["datetime"].dt.weekday   # Monday=0
df["is_weekend"] = df["weekday"].isin([5, 6]).astype(int)

In [25]:
def time_of_day(hour):
    if 5 <= hour < 12:
        return "Morning"
    elif 12 <= hour < 17:
        return "Afternoon"
    elif 17 <= hour < 21:
        return "Evening"
    else:
        return "Night"

df["time_of_day"] = df["hour"].apply(time_of_day)

In [26]:
df["time_of_day"].value_counts()

time_of_day
Night        6724
Morning      5734
Afternoon    4180
Evening      3362
Name: count, dtype: int64

In [27]:
def season(month):
    if month in [12, 1, 2]:
        return "Winter"
    elif month in [3, 4, 5]:
        return "Summer"
    elif month in [6, 7, 8, 9]:
        return "Monsoon"
    else:
        return "Post-Monsoon"

df["season"] = df["month"].apply(season)

In [28]:
df["season"].value_counts()

season
Monsoon         6118
Winter          5434
Summer          5399
Post-Monsoon    3049
Name: count, dtype: int64

In [30]:
cat_cols = df.select_dtypes(include="object").columns

for col in cat_cols:
    df[col] = (
        df[col]
        .fillna("Unknown")
        .astype(str)
        .str.strip()
    )


In [31]:
if "Severity" in df.columns:
    df["Severity"] = (
        df["Severity"]
        .astype(str)
        .str.lower()
        .str.strip()
    )

In [32]:
df.info

<bound method DataFrame.info of        accident_id        city        state   latitude  longitude       date  \
0                0        Pune  Maharashtra  18.680827  73.930388 2023-10-22   
1                1      Mumbai  Maharashtra  18.817732  72.790846 2023-05-21   
2                2      Mumbai  Maharashtra  19.096889  72.819424 2024-07-10   
3                3  Chandigarh       Punjab  30.787805  76.847507 2025-03-30   
4                4     Chennai   Tamil Nadu  12.965155  80.283313 2024-01-25   
...            ...         ...          ...        ...        ...        ...   
19995        19995   Bangalore    Karnataka  13.092276  77.599571 2022-09-29   
19996        19996     Chennai   Tamil Nadu  13.172928  80.157062 2023-11-25   
19997        19997     Chennai   Tamil Nadu  12.997170  80.150724 2022-06-18   
19998        19998     Kolkata  West Bengal  22.454882  88.322213 2023-03-12   
19999        19999       Delhi        Delhi  28.510266  77.065301 2024-07-05   

       

In [34]:
import os

os.makedirs("../data/processed", exist_ok=True)

In [35]:
df.to_csv("../data/processed/cleaned_accident_data.csv", index=False)